# Data Cleaning & Reporting Automation
## Hinglish walkthrough

Is notebook mein hum raw sales dataset ko inspect, clean aur analyze karenge. Saath hi CSV reports aur charts generate honge.

**Run order:** cells ko top-to-bottom run karo. Project root folder se Jupyter start karna best hai.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from data_cleaning import load_data, clean_sales_data
from generate_report import build_reports

RAW_PATH = ROOT / "data" / "raw_sales_data.csv"
REPORTS_DIR = ROOT / "reports"
print("Project root:", ROOT)
print("Raw dataset exists:", RAW_PATH.exists())

## 1. Raw data load karo

Pehle dataset ke first rows, shape aur data types check karte hain. Isse pata chalega data mein kaun-se columns hain.

In [ ]:
raw = load_data(RAW_PATH)
display(raw.head())
print("Rows, columns:", raw.shape)
display(raw.dtypes.to_frame("data_type"))
display(raw.isna().sum().sort_values(ascending=False).to_frame("missing_values"))

## 2. Data quality issues identify karo

Yahan missing cells, exact duplicates, aur invalid quantities count karte hain.

In [ ]:
print("Exact duplicate rows:", raw.duplicated().sum())
print("Missing cells:", raw.isna().sum().sum())
if "Quantity" in raw.columns:
    print("Non-positive quantities:", (pd.to_numeric(raw["Quantity"], errors="coerce") <= 0).sum())

## 3. Data clean karo

Rules:
- Duplicate rows remove
- Text whitespace aur casing normalize
- Invalid dates/numbers handle
- Missing prices median se fill
- Missing discount ko 0 set
- Quantity invalid ho to transaction remove
- Sales_Amount dobara calculate

Business ke real data par apply karne se pehle imputation rules confirm karna zaroori hai.

In [ ]:
cleaned, metrics = clean_sales_data(raw)
display(cleaned.head())
display(pd.DataFrame({"Metric": list(metrics.keys()), "Value": list(metrics.values())}))
print("Cleaned shape:", cleaned.shape)

## 4. Automated reporting run karo

Ye function cleaned CSV, summaries, executive summary aur PNG charts create karega.

In [ ]:
cleaned, metrics, summary = build_reports(RAW_PATH, REPORTS_DIR)
print("Executive summary:")
for key, value in summary.items():
    print(f"{key}: {value}")
print("\nReports folder:", REPORTS_DIR)

## 5. Monthly sales trend

In [ ]:
monthly = pd.read_csv(REPORTS_DIR / "monthly_sales_summary.csv")
display(monthly.head())
plt.figure(figsize=(12, 5))
plt.plot(monthly["Month"], monthly["Total_Sales"], marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales (INR)")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

## 6. Product aur region performance

In [ ]:
products = pd.read_csv(REPORTS_DIR / "product_sales_summary.csv")
regions = pd.read_csv(REPORTS_DIR / "region_sales_summary.csv")
display(products)
display(regions)

plt.figure(figsize=(9, 5))
plt.bar(products["Product"], products["Total_Sales"])
plt.title("Sales by Product")
plt.xlabel("Product")
plt.ylabel("Sales (INR)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.bar(regions["Region"], regions["Total_Sales"])
plt.title("Sales by Region")
plt.xlabel("Region")
plt.ylabel("Sales (INR)")
plt.tight_layout()
plt.show()

## 7. Cleaned data export verify karo

Power BI mein import karne ke liye cleaned CSV use hogi.

In [ ]:
cleaned_path = ROOT / "data" / "cleaned_sales_data.csv"
final_df = pd.read_csv(cleaned_path)
print("Saved file:", cleaned_path)
print("Saved shape:", final_df.shape)
print("Missing cells after cleaning:", final_df.isna().sum().sum())
display(final_df.head())

## 8. Power BI next steps

1. Power BI Desktop open karo.
2. Get data → Text/CSV → `data/cleaned_sales_data.csv`.
3. `dashboard/power_bi_dashboard_guide.md` se DAX measures copy karo.
4. KPI cards, monthly trend, product/region charts aur slicers add karo.
5. File ko `dashboard/sales_reporting_dashboard.pbix` naam se save karo.

**Note:** PBIX Power BI Desktop mein create hoti hai; Python ZIP mein ready guide aur cleaned dataset included hain.